In [36]:
import pandas as pd
from pathlib import Path
import gc

from schema import SCHEMA_CURICA

In [37]:
# toma como diretório pai o diretório do arquivo do notebook
BASE_DIR = Path("etl_censo.ipynb").resolve().parent
RAW_CENSO_DIR = BASE_DIR / "data" / "raw"
RAW_ANEXOS = RAW_CENSO_DIR / "anexos"

In [38]:
df_escola_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Escola_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_matricula_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Matricula_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_turma_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Turma_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_docente_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Docente_2025.csv", sep=";", encoding="latin-1", low_memory=False)

df_escola_ac_2025 = df_escola_2025[df_escola_2025["SG_UF"] == "AC"].copy()
df_matricula_ac_2025 = df_matricula_2025[df_matricula_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()
df_turma_ac_2025 = df_turma_2025[df_turma_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()
df_docente_ac_2025 =  df_docente_2025[df_docente_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()

del df_escola_2025
del df_matricula_2025
del df_turma_2025
del df_docente_2025

gc.collect()

1359

In [39]:
df_escola_ac_2025 = df_escola_ac_2025[df_escola_ac_2025["TP_SITUACAO_FUNCIONAMENTO"] == 1].copy()

In [40]:
df_censo_ac_2025 = df_escola_ac_2025.merge(
    df_matricula_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_mat")
)

df_censo_ac_2025 = df_censo_ac_2025.merge(
    df_turma_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_tur")
)

df_censo_ac_2025 = df_censo_ac_2025.merge(
    df_docente_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_doc")
)

In [41]:
del df_escola_ac_2025
del df_matricula_ac_2025
del df_turma_ac_2025
del df_docente_ac_2025

gc.collect()

0

In [42]:
df_censo_ac_2025 = df_censo_ac_2025[df_censo_ac_2025['TP_DEPENDENCIA'] == 2]

df_censo_ac_2025 = df_censo_ac_2025[SCHEMA_CURICA]

df_censo_ac_2025.shape

(592, 323)

In [43]:
df_anexos = pd.read_csv(RAW_ANEXOS / "anexos_escolas_estaduais.csv", sep=";", encoding="latin-1", low_memory=False)



In [44]:
df_anexos = df_anexos.rename(columns={'Código da Escola': 'CO_ENTIDADE'})



In [45]:
df_anexos.columns

Index([' MUNICÍPIO ', 'CO_ENTIDADE', 'ESCOLA ', 'Localização',
       'MODALIDADE DE ENSINO', 'ENDEREÇOS ', 'ATIVA(1)'],
      dtype='str')

In [46]:
df_anexos['CO_ENTIDADE'].nunique()

565

In [47]:
codigos_com_anexos = df_anexos['CO_ENTIDADE'].value_counts()
codigos_com_anexos = codigos_com_anexos[codigos_com_anexos > 1].index

df_censo_ac_2025['tem_anexo'] = df_censo_ac_2025['CO_ENTIDADE'].isin(codigos_com_anexos).astype(int)

df_censo_ac_2025['tem_anexo'].value_counts()

tem_anexo
0    519
1     73
Name: count, dtype: int64

Index(['IN_ENERGIA_REDE_PUBLICA', 'QT_MAT_INF_INT', 'QT_MAT_INF_CRE_INT',
       'QT_MAT_INF_PRE_INT', 'QT_MAT_FUND_INT', 'QT_MAT_FUND_AI_INT',
       'QT_MAT_FUND_AF_INT', 'QT_MAT_MED_INT'],
      dtype='str')

In [48]:
# coluna target
y = df_censo_ac_2025['tem_anexo'] 

features = [SCHEMA_CURICA]

X = df_censo_ac_2025

X = X.fillna(0)

X = X.loc[:, ~X.columns.duplicated()]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [49]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

# Transformador
preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# Pipeline completo
pipeline = Pipeline(steps=[
    ('preprocessamento', preprocessador),
    ('modelo', LogisticRegression(max_iter=1000))
])

# Treino
pipeline.fit(X_train, y_train)

# Predição
y_pred = pipeline.predict(X_test)

/tmp/ipykernel_55078/4154483002.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns


TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['int', 'str']

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))